# Adaptive Feedback-to-Action Coach

This notebook demonstrates the first coding milestone: a Cellmate-compatible decision layer. It shows how a hidden-test result from a notebook feedback workflow can be converted into an adaptive support format and a next learning action.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "adaptive_coach").exists() and (root.parent / "adaptive_coach").exists():
    root = root.parent
sys.path.insert(0, str(root))

from adaptive_coach import recommendation_from_cellmate_event, recommendation_to_markdown
from IPython.display import Markdown, clear_output, display

## Scenario 1: Cellmate-style hidden test event

Cellmate already runs notebook code against hidden tests. This cell uses a pytest-json-report style event like the one produced after a failed hidden-test run.

In [ ]:
cellmate_event = {
    "exerciseId": "sum_numbers",
    "code": """
def sum_numbers(values):
    total = 0
    for value in values:
        total = value
    return total
""",
    "attemptCount": 3,
    "previousMistakeTypes": ["accumulator_update_error"],
    "testResult": {
        "stdout": "",
        "stderr": "",
        "timeout": False,
        "report": {
            "tests": [
                {
                    "nodeid": "test_hidden.py::test_positive_list",
                    "outcome": "failed",
                    "call": {
                        "longrepr": {
                            "reprcrash": {"message": "assert 4 == 10"}
                        }
                    },
                },
                {
                    "nodeid": "test_hidden.py::test_mixed_list",
                    "outcome": "failed",
                    "call": {
                        "longrepr": {
                            "reprcrash": {"message": "assert 4 == 5"}
                        }
                    },
                },
                {
                    "nodeid": "test_hidden.py::test_empty_list",
                    "outcome": "passed",
                },
            ]
        },
    },
    "metadata": {"concepts": ["loops", "accumulator", "variable-update"]},
}

recommendation = recommendation_from_cellmate_event(cellmate_event)

print("Cellmate hidden-test evidence")
for test in cellmate_event["testResult"]["report"]["tests"]:
    message = test.get("call", {}).get("longrepr", {}).get("reprcrash", {}).get("message", "passed")
    print(f"- {test['nodeid']}: {test['outcome']} ({message})")

print("\nAdaptive decision")
print("correct:", recommendation["correct"])
print("mistake_type:", recommendation["mistake_type"])
print("support_format:", recommendation["support_format"])
print("next_action:", recommendation["next_action"])

Cellmate hidden-test evidence
- test_hidden.py::test_positive_list: failed (assert 4 == 10)
- test_hidden.py::test_mixed_list: failed (assert 4 == 5)
- test_hidden.py::test_empty_list: passed (passed)

Adaptive decision
correct: False
mistake_type: accumulator_update_error
support_format: state_update_table
next_action: easier_intermediate_exercise


## Rendered notebook feedback

The same recommendation can be rendered as a notebook feedback cell that Cellmate could insert below the submitted code cell. The generated exercise should then be inserted as a real code cell, not only as a Markdown code block.

In [ ]:
display(Markdown(recommendation_to_markdown(recommendation)))

### Adaptive coach: Needs work

- **Mistake type:** `accumulator_update_error`
- **Support format:** `state_update_table`
- **Next action:** `easier_intermediate_exercise`

#### Support
The running total is being replaced each time instead of updated.

| iteration | value | total before | current update | total after |
|---:|---:|---:|---|---:|
| 1 | 1 | 0 | total = value | 1 |
| 2 | 2 | 1 | total = value | 2 |
| 3 | 3 | 2 | total = value | 3 |
| 4 | 4 | 3 | total = value | 4 |

The previous total should be kept and combined with the current value.

#### Generated follow-up exercise
**Intermediate: update a running total**

Complete the update line so total becomes 5 after adding 2 and 3.

```python
total = 0
for value in [2, 3]:
    # update total so it becomes 5
```

In [ ]:
%pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   ---------------------- ----------------- 524.3/914.9 kB 1.8 MB/s eta 0:00:01
   ---------------------------------------- 914.9/914.9 kB 1.7 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Dynamic next-step workflow

This cell keeps the workflow in one changing output area. The student only sees the current recommended action; after each check, the display updates to the next step.

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None


def _normalise_answer(text):
    return text.replace(" ", "").strip()


def _workflow_markdown(stage, observed=None):
    if stage == "intermediate":
        return (
            "### Current next step: easier intermediate exercise\n"
            "The student has repeated the accumulator update mistake, so the system "
            "does not unlock a harder task yet.\n\n"
            "Complete the update line so `total` becomes `5` after adding `2` and `3`."
        )
    if stage == "retry_original":
        return (
            "### Intermediate exercise passed\n"
            "Next action: retry the original summation task using the same update pattern."
        )
    if stage == "harder_extension":
        return (
            "### Original task passed\n"
            "Next action: generate a harder extension task that combines accumulation "
            "with conditional filtering."
        )
    return (
        "### Still needs targeted support\n"
        f"Observed result: `{observed}`. The system keeps the student on the current step."
    )


if widgets is None:
    display(Markdown(_workflow_markdown("intermediate")))
    display(Markdown(
        "`ipywidgets` is not available, so this notebook shows the dynamic workflow "
        "as text. In VS Code/Jupyter with widgets enabled, this cell becomes clickable."
    ))
else:
    output = widgets.Output()
    intermediate_answer = widgets.Text(
        value="total = total + value",
        description="Update line:",
        layout=widgets.Layout(width="520px"),
    )
    retry_answer = widgets.Text(
        value="total = total + value",
        description="Retry line:",
        layout=widgets.Layout(width="520px"),
    )
    extension_answer = widgets.Text(
        value="value > 0",
        description="Condition:",
        layout=widgets.Layout(width="520px"),
    )

    check_intermediate = widgets.Button(description="Check intermediate", button_style="primary")
    check_retry = widgets.Button(description="Check original retry", button_style="primary")
    check_extension = widgets.Button(description="Check extension", button_style="success")

    def render(stage, observed=None):
        with output:
            clear_output(wait=True)
            display(Markdown(_workflow_markdown(stage, observed)))
            if stage == "intermediate":
                display(Markdown("```python\ntotal = 0\nfor value in [2, 3]:\n    # write the update line\ntotal\n```"))
                display(intermediate_answer, check_intermediate)
            elif stage == "retry_original":
                display(Markdown("```python\ndef sum_numbers(values):\n    total = 0\n    for value in values:\n        # write the update line\n    return total\n```"))
                display(retry_answer, check_retry)
            elif stage == "harder_extension":
                display(Markdown("```python\ndef sum_positive(values):\n    total = 0\n    for value in values:\n        if ...:\n            total = total + value\n    return total\n```"))
                display(extension_answer, check_extension)

    def on_check_intermediate(_):
        answer = _normalise_answer(intermediate_answer.value)
        if answer in {"total=total+value", "total+=value"}:
            render("retry_original")
        else:
            render("needs_support", observed=intermediate_answer.value)

    def on_check_retry(_):
        answer = _normalise_answer(retry_answer.value)
        if answer in {"total=total+value", "total+=value"}:
            render("harder_extension")
        else:
            render("needs_support", observed=retry_answer.value)

    def on_check_extension(_):
        answer = _normalise_answer(extension_answer.value)
        if answer in {"value>0", "0<value"}:
            with output:
                clear_output(wait=True)
                display(Markdown("### Extension passed\nThe student is ready to move on."))
        else:
            render("needs_support", observed=extension_answer.value)

    check_intermediate.on_click(on_check_intermediate)
    check_retry.on_click(on_check_retry)
    check_extension.on_click(on_check_extension)

    display(output)
    render("intermediate")

Output()

## Scenario 2: successful retry

Once the accumulator is fixed, the system recommends a harder extension.

In [ ]:
event = {
    "exercise_id": "sum_numbers",
    "student_code": """
def sum_numbers(values):
    total = 0
    for value in values:
        total = total + value
    return total
""",
    "attempt_count": 4,
    "previous_mistake_types": ["accumulator_update_error"],
}

recommendation = recommendation_from_cellmate_event(event)
display(Markdown(recommendation_to_markdown(recommendation)))

### Adaptive coach: Correct

- **Mistake type:** `none`
- **Support format:** `short_confirmation`
- **Next action:** `harder_extension`

#### Support
The hidden tests passed. The student is ready for a harder follow-up task.

#### Generated follow-up exercise
**Extension: sum positive numbers**

Write sum_positive(values). Only values greater than zero should be added.

```python
def sum_positive(values):
    total = 0
    for value in values:
        # add only positive values
        pass
    return total
```

## Scenario 3: conditional filtering error

The student knows the accumulator pattern, but now adds every value instead of only positive values.

In [ ]:
event = {
    "exercise_id": "sum_positive",
    "student_code": """
def sum_positive(values):
    total = 0
    for value in values:
        total = total + value
    return total
""",
    "attempt_count": 1,
    "previous_mistake_types": [],
}

recommendation = recommendation_from_cellmate_event(event)
display(Markdown(recommendation_to_markdown(recommendation)))

### Adaptive coach: Needs work

- **Mistake type:** `conditional_filtering_error`
- **Support format:** `include_exclude_comparison`
- **Next action:** `retry_with_targeted_support`

#### Support
The loop is adding values that should be excluded by the condition.

| value | should include? | reason |
|---:|---|---|
| -2 | no | not positive |
| 3 | yes | positive |
| 4 | yes | positive |

The update should happen only inside the branch for positive values.